# Émotion — Donner une âme au message

Ce notebook analyse les émotions de chaque message du corpus pour construire un **profil émotionnel** qui pilotera :
- la **voix** de l'avatar (prosodie, rythme, énergie)
- le **visage** de l'avatar (expressions faciales, regard)
- le **comportement** de l'avatar (dynamique d'interaction)

### Approche

On combine deux méthodes complémentaires :

1. **Modèle NLP** — Sentiment multilingual (XLM-RoBERTa) + émotions (DistilRoBERTa) pour une analyse globale
2. **Lexique contextuel** — Dimensions émotionnelles spécifiques à ce corpus de solidarité et d'accueil

### Pipeline
```
Témoignage texte
      ↓
┌─────────────────────────────────────┐
│ Sentiment (positif/négatif/neutre)  │ ← XLM-RoBERTa
│ Émotions (joy/sadness/anger/fear…)  │ ← DistilRoBERTa
│ Dimensions contextuelles            │ ← Lexique solidarité
└─────────────────────────────────────┘
      ↓
Profil émotionnel
      ↓
┌──────────────┐  ┌──────────────┐
│ Params voix  │  │ Params visage│
│ pitch, speed │  │ sourcils,    │
│ energy, warm │  │ bouche, yeux │
└──────────────┘  └──────────────┘
```

**Input** : `data/processed/corpus.parquet`  
**Output** : `data/processed/corpus_with_emotions.parquet` — messages annotés + paramètres avatar

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, HTML
from transformers import pipeline
from tqdm.auto import tqdm
import re
import json
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size': 12, 'figure.facecolor': 'white'})
sns.set_palette('Set2')

ModuleNotFoundError: No module named 'transformers'

In [ ]:
# Charger le corpus propre
corpus = pd.read_parquet('../data/processed/corpus.parquet')
print(f'Corpus : {len(corpus):,} messages')
print(f'Avec témoignage : {corpus["testimony"].notna().sum():,}')

---
## 1. Analyse de sentiment (XLM-RoBERTa)

Sentiment global : le message est-il positif, négatif, ou neutre ?

Utilise un modèle multilingue entraîné sur 8 langues, dont le français.

In [ ]:
# Charger le modèle de sentiment
print('Chargement du modèle de sentiment...')
sentiment_pipe = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual',
    device=-1,  # CPU
    truncation=True,
    max_length=512
)
print('✓ Modèle sentiment chargé')

In [ ]:
# Analyser le sentiment de chaque message
# On prend le testimony en priorité, sinon le full_text
texts_for_analysis = corpus['testimony'].fillna(corpus['full_text']).fillna('').tolist()

# Traiter par batch pour la performance
BATCH = 32
sentiments = []

for i in tqdm(range(0, len(texts_for_analysis), BATCH), desc='Sentiment'):
    batch = texts_for_analysis[i:i+BATCH]
    # Tronquer les textes très longs
    batch = [t[:1500] if len(t) > 1500 else t for t in batch]
    results = sentiment_pipe(batch)
    sentiments.extend(results)

corpus['sentiment_label'] = [s['label'] for s in sentiments]
corpus['sentiment_score'] = [s['score'] for s in sentiments]

print(f'\nDistribution des sentiments :')
print(corpus['sentiment_label'].value_counts())

In [ ]:
# Visualiser
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'positive': '#2ecc71', 'neutral': '#f39c12', 'negative': '#e74c3c'}
counts = corpus['sentiment_label'].value_counts()

axes[0].pie(counts, labels=counts.index, autopct='%1.0f%%',
            colors=[colors.get(l, '#95a5a6') for l in counts.index],
            startangle=90, textprops={'fontsize': 13})
axes[0].set_title('Sentiment des témoignages', fontsize=14)

corpus['sentiment_score'].hist(bins=40, ax=axes[1], color='#3498db', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Confiance du modèle')
axes[1].set_title('Distribution de la confiance', fontsize=14)

plt.tight_layout()
plt.show()

---
## 2. Détection d'émotions (DistilRoBERTa)

7 émotions : **anger, disgust, fear, joy, neutral, sadness, surprise**

Le modèle est en anglais — on traduit à la volée les messages courts ou on utilise les patterns linguistiques cross-lingues du modèle.

In [ ]:
# Charger le modèle d'émotions
print('Chargement du modèle d\'émotions...')
emotion_pipe = pipeline(
    'text-classification',
    model='j-hartmann/emotion-english-distilroberta-base',
    top_k=None,
    device=-1,
    truncation=True,
    max_length=512
)
print('✓ Modèle émotions chargé')

In [ ]:
# Analyser les émotions
emotions_all = []

for i in tqdm(range(0, len(texts_for_analysis), BATCH), desc='Émotions'):
    batch = texts_for_analysis[i:i+BATCH]
    batch = [t[:1500] if len(t) > 1500 else t for t in batch]
    results = emotion_pipe(batch)
    emotions_all.extend(results)

# Extraire les scores d'émotion pour chaque message
emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

for label in emotion_labels:
    corpus[f'emo_{label}'] = [
        next((e['score'] for e in msg if e['label'] == label), 0.0)
        for msg in emotions_all
    ]

# Émotion dominante
emo_cols = [f'emo_{l}' for l in emotion_labels]
corpus['dominant_emotion'] = corpus[emo_cols].idxmax(axis=1).str.replace('emo_', '')

print(f'\nÉmotion dominante :')
print(corpus['dominant_emotion'].value_counts())

In [ ]:
# Heatmap des émotions moyennes par source
emo_by_source = corpus.groupby('source')[emo_cols].mean()
emo_by_source.columns = emotion_labels

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribution des émotions dominantes
emo_colors = {
    'joy': '#f1c40f', 'sadness': '#3498db', 'anger': '#e74c3c',
    'fear': '#9b59b6', 'surprise': '#e67e22', 'neutral': '#95a5a6', 'disgust': '#1abc9c'
}
dom_counts = corpus['dominant_emotion'].value_counts()
dom_counts.plot(kind='bar', ax=axes[0], color=[emo_colors.get(e, '#333') for e in dom_counts.index], edgecolor='white')
axes[0].set_title('Émotion dominante dans le corpus', fontsize=14)
axes[0].set_ylabel('Messages')

# Scores moyens
mean_emos = corpus[emo_cols].mean().sort_values(ascending=True)
mean_emos.index = [l.replace('emo_', '') for l in mean_emos.index]
mean_emos.plot(kind='barh', ax=axes[1], color=[emo_colors.get(e, '#333') for e in mean_emos.index], edgecolor='white')
axes[1].set_title('Score moyen par émotion', fontsize=14)
axes[1].set_xlabel('Score moyen')

plt.tight_layout()
plt.show()

---
## 3. Dimensions émotionnelles contextuelles

Les émotions génériques (joy, sadness...) ne capturent pas la richesse émotionnelle spécifique de ce corpus.

On construit un **lexique contextuel** avec des dimensions propres à l'accueil de réfugiés :

| Dimension | Ce qu'elle capte | Effet avatar |
|-----------|------------------|--------------|
| **Solidarité** | Engagement collectif, entraide | Regard direct, voix ferme et chaleureuse |
| **Empathie** | Compassion, compréhension de la souffrance | Regard doux, voix plus lente et posée |
| **Urgence** | Choc, besoin d'agir maintenant | Sourcils relevés, débit plus rapide |
| **Chaleur** | Accueil, partage, famille | Sourire, voix chaleureuse et enveloppante |
| **Tristesse** | Émotion face à la souffrance | Regard baissé, voix basse, pauses |
| **Espoir** | Optimisme, croire au changement | Yeux lumineux, voix montante |
| **Détermination** | Décision ferme, volonté d'agir | Mâchoire serrée, voix posée et décidée |
| **Indignation** | Colère face à l'injustice | Sourcils froncés, voix plus forte |

In [ ]:
# Lexique contextuel — dimensions émotionnelles spécifiques au corpus
LEXICON = {
    'solidarite': [
        'solidarité', 'solidaire', 'citoyen', 'citoyenne', 'ensemble', 'collectif',
        'communauté', 'responsabilité', 'devoir', 'contribuer', 'engagement', 'engagé',
        'engagée', 'mobiliser', 'agir', 'action', 'faire quelque chose', 'concrètement',
        'participer', 'bras ballants', 'rester sans rien faire'
    ],
    'empathie': [
        'comprendre', 'comprends', 'touché', 'touchée', 'ému', 'émue', 'sensible',
        'compassion', 'souffrance', 'douleur', 'difficile', 'épreuve', 'parcours',
        'imagine', 'imaginer', 'place', 'à leur place', 'humain', 'humanité',
        'dignité', 'humaniste', 'prochain'
    ],
    'urgence': [
        'choqué', 'choquée', 'choc', 'urgence', 'urgent', 'insupportable',
        'inacceptable', 'inadmissible', 'catastrophe', 'crise', 'détresse',
        'immédiatement', 'maintenant', 'vite', 'ne plus attendre', 'assez',
        'trop', 'images', 'médias', 'télévision', 'actualité'
    ],
    'chaleur': [
        'accueil', 'accueillir', 'maison', 'chez nous', 'chez moi', 'foyer',
        'chambre', 'partage', 'partager', 'repas', 'ensemble', 'famille',
        'amitié', 'convivialité', 'ouvert', 'ouverte', 'bienvenue',
        'chaleureux', 'chaleureuse', 'bienveillance', 'tendresse'
    ],
    'tristesse': [
        'triste', 'tristesse', 'pleure', 'pleurer', 'larmes', 'désolé', 'désolée',
        'malheureux', 'malheureuse', 'terrible', 'drame', 'tragédie', 'tragique',
        'mort', 'mourir', 'noyade', 'noyé', 'aylan', 'guerre', 'bombe',
        'destruction', 'perdu', 'perdue', 'déraciné'
    ],
    'espoir': [
        'espoir', 'espérer', 'espère', 'avenir', 'futur', 'possible', 'possible',
        'mieux', 'meilleur', 'chance', 'opportunité', 'reconstruire', 'reconstruction',
        'nouveau départ', 'recommencer', 'rêve', 'rêver', 'croire', 'confiance',
        'optimiste', 'positif', 'positive'
    ],
    'determination': [
        'décidé', 'décidée', 'décision', 'prêt', 'prête', 'prêts', 'volonté',
        'résolu', 'résolution', 'fermement', 'convaincu', 'convaincue',
        'évident', 'naturel', 'logique', 'normal', 'bien sûr', 'évidemment',
        'sans hésiter', 'absolument', 'totalement'
    ],
    'indignation': [
        'révolte', 'révolté', 'révoltée', 'scandale', 'scandaleux',
        'honte', 'honteux', 'intolérable', 'injuste', 'injustice',
        'colère', 'furieux', 'furieuse', 'indigné', 'indignée',
        'politique', 'gouvernement', 'laisser faire', 'abandon'
    ]
}

def score_dimensions(text):
    """Calcule un score [0,1] pour chaque dimension émotionnelle."""
    if pd.isna(text) or not isinstance(text, str):
        return {dim: 0.0 for dim in LEXICON}
    
    text_lower = text.lower()
    scores = {}
    
    for dim, keywords in LEXICON.items():
        hits = sum(1 for kw in keywords if kw in text_lower)
        # Normaliser : sigmoid-like pour éviter la saturation
        scores[dim] = 1 - (1 / (1 + hits * 0.5))
    
    return scores

# Appliquer
dim_scores = corpus['testimony'].fillna(corpus['full_text']).apply(score_dimensions)
dim_df = pd.DataFrame(dim_scores.tolist())

for dim in LEXICON:
    corpus[f'dim_{dim}'] = dim_df[dim].values

# Dimension dominante
dim_cols = [f'dim_{d}' for d in LEXICON]
corpus['dominant_dimension'] = corpus[dim_cols].idxmax(axis=1).str.replace('dim_', '')

print('Dimension contextuelle dominante :')
print(corpus['dominant_dimension'].value_counts())

In [ ]:
# Radar chart des dimensions émotionnelles moyennes
dims = list(LEXICON.keys())
values = [corpus[f'dim_{d}'].mean() for d in dims]

# Radar
angles = np.linspace(0, 2 * np.pi, len(dims), endpoint=False).tolist()
values_plot = values + [values[0]]  # fermer le polygone
angles += [angles[0]]
dims_plot = dims + [dims[0]]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), subplot_kw={'polar': True} if True else {})

# Radar global
ax = plt.subplot(121, polar=True)
ax.fill(angles, values_plot, alpha=0.25, color='#3498db')
ax.plot(angles, values_plot, 'o-', linewidth=2, color='#3498db')
ax.set_xticks(angles[:-1])
ax.set_xticklabels([d.capitalize() for d in dims], fontsize=11)
ax.set_title('Profil émotionnel moyen du corpus', fontsize=14, pad=20)

# Bar chart dimensions
ax2 = plt.subplot(122)
dim_means = pd.Series({d.capitalize(): corpus[f'dim_{d}'].mean() for d in dims}).sort_values()
dim_colors = ['#e74c3c', '#9b59b6', '#3498db', '#1abc9c', '#f39c12', '#2ecc71', '#e67e22', '#f1c40f']
dim_means.plot(kind='barh', ax=ax2, color=dim_colors[:len(dim_means)], edgecolor='white')
ax2.set_title('Score moyen par dimension', fontsize=14)
ax2.set_xlabel('Score')

plt.tight_layout()
plt.show()

---
## 4. Mapping vers l'avatar

Transformer le profil émotionnel en paramètres concrets pour la voix et le visage.

In [ ]:
# Mapping : dimension émotionnelle → paramètres avatar

# VOIX : pitch (-1 grave, +1 aigu), speed (0.5 lent, 1.5 rapide),
#         energy (0 calme, 1 intense), warmth (0 froid, 1 chaleureux)
VOICE_MAP = {
    'solidarite':    {'pitch': 0.1,  'speed': 1.05, 'energy': 0.7,  'warmth': 0.7},
    'empathie':      {'pitch': -0.1, 'speed': 0.85, 'energy': 0.4,  'warmth': 0.9},
    'urgence':       {'pitch': 0.3,  'speed': 1.2,  'energy': 0.85, 'warmth': 0.3},
    'chaleur':       {'pitch': 0.0,  'speed': 0.9,  'energy': 0.5,  'warmth': 1.0},
    'tristesse':     {'pitch': -0.3, 'speed': 0.75, 'energy': 0.3,  'warmth': 0.6},
    'espoir':        {'pitch': 0.2,  'speed': 1.0,  'energy': 0.6,  'warmth': 0.8},
    'determination': {'pitch': 0.0,  'speed': 0.95, 'energy': 0.7,  'warmth': 0.5},
    'indignation':   {'pitch': 0.4,  'speed': 1.15, 'energy': 0.9,  'warmth': 0.2},
}

# VISAGE : eyebrows (-1 froncés, +1 relevés), mouth (-1 triste, +1 sourire),
#           eyes (0 mi-clos, 1 grands ouverts), gaze (0 baissé, 1 direct)
FACE_MAP = {
    'solidarite':    {'eyebrows': 0.3,  'mouth': 0.3,  'eyes': 0.8, 'gaze': 1.0},
    'empathie':      {'eyebrows': 0.1,  'mouth': 0.0,  'eyes': 0.6, 'gaze': 0.8},
    'urgence':       {'eyebrows': 0.6,  'mouth': -0.2, 'eyes': 1.0, 'gaze': 1.0},
    'chaleur':       {'eyebrows': 0.2,  'mouth': 0.6,  'eyes': 0.7, 'gaze': 0.9},
    'tristesse':     {'eyebrows': -0.3, 'mouth': -0.5, 'eyes': 0.3, 'gaze': 0.3},
    'espoir':        {'eyebrows': 0.4,  'mouth': 0.4,  'eyes': 0.9, 'gaze': 0.7},
    'determination': {'eyebrows': -0.1, 'mouth': 0.0,  'eyes': 0.7, 'gaze': 1.0},
    'indignation':   {'eyebrows': -0.5, 'mouth': -0.3, 'eyes': 0.9, 'gaze': 1.0},
}

def compute_avatar_params(row):
    """Calcule les paramètres avatar par interpolation pondérée des dimensions."""
    voice = {'pitch': 0, 'speed': 1.0, 'energy': 0.5, 'warmth': 0.5}
    face = {'eyebrows': 0, 'mouth': 0, 'eyes': 0.5, 'gaze': 0.5}
    
    total_weight = 0
    for dim in LEXICON:
        w = row[f'dim_{dim}']
        if w > 0.01:
            total_weight += w
            for param in voice:
                voice[param] += w * VOICE_MAP[dim][param]
            for param in face:
                face[param] += w * FACE_MAP[dim][param]
    
    if total_weight > 0:
        for param in voice:
            voice[param] /= total_weight
        for param in face:
            face[param] /= total_weight
    
    return {**{f'voice_{k}': v for k, v in voice.items()},
            **{f'face_{k}': v for k, v in face.items()}}

# Appliquer
avatar_params = corpus.apply(compute_avatar_params, axis=1)
params_df = pd.DataFrame(avatar_params.tolist())

for col in params_df.columns:
    corpus[col] = params_df[col].values

print('✓ Paramètres avatar calculés')
print(f'\nParamètres voix (moyennes) :')
for p in ['voice_pitch', 'voice_speed', 'voice_energy', 'voice_warmth']:
    print(f'  {p:15s}: {corpus[p].mean():.3f}')
print(f'\nParamètres visage (moyennes) :')
for p in ['face_eyebrows', 'face_mouth', 'face_eyes', 'face_gaze']:
    print(f'  {p:15s}: {corpus[p].mean():.3f}')

---
## 5. Résultat : messages prêts pour l'avatar

In [ ]:
def show_emotion_card(row):
    """Affiche un message avec son profil émotionnel complet."""
    
    # Couleurs par sentiment
    sent_color = {'positive': '#27ae60', 'neutral': '#f39c12', 'negative': '#c0392b'}.get(row['sentiment_label'], '#999')
    
    # Dimensions contextuelles triées
    dims_sorted = sorted(LEXICON.keys(), key=lambda d: row[f'dim_{d}'], reverse=True)
    dims_html = ' '.join(
        f"<span style='background:#ecf0f1; padding:2px 8px; border-radius:10px; margin:2px; font-size:12px;'>"
        f"{d} <b>{row[f'dim_{d}']:.2f}</b></span>"
        for d in dims_sorted[:4] if row[f'dim_{d}'] > 0.01
    )
    
    # Paramètres avatar
    def bar(val, low=-1, high=1):
        pct = (val - low) / (high - low) * 100
        return f"<div style='background:#ecf0f1; height:8px; border-radius:4px; width:120px; display:inline-block;'><div style='background:#3498db; height:8px; border-radius:4px; width:{pct:.0f}%;'></div></div>"
    
    html = f"""
    <div style='border:1px solid #ddd; border-radius:8px; padding:16px; margin:16px 0; background:white; box-shadow:0 1px 3px rgba(0,0,0,0.1);'>
        <div style='display:flex; justify-content:space-between; align-items:center; margin-bottom:10px;'>
            <span style='font-weight:bold; color:#2c3e50;'>{row['id']}</span>
            <span style='background:{sent_color}; color:white; padding:3px 10px; border-radius:12px; font-size:12px;'>
                {row['sentiment_label']} ({row['sentiment_score']:.0%})
            </span>
        </div>
    """
    
    if pd.notna(row['testimony']):
        text = row['testimony'][:400].replace('\n', '<br>')
        if len(row['testimony']) > 400: text += '…'
        html += f"<div style='font-style:italic; color:#34495e; margin:10px 0; line-height:1.5;'>❝ {text} ❞</div>"
    
    html += f"""
        <div style='margin-top:12px;'>
            <div style='font-size:12px; color:#7f8c8d; margin-bottom:4px;'>Émotion dominante : <b>{row['dominant_emotion']}</b> · Dimension : <b>{row['dominant_dimension']}</b></div>
            <div style='margin:6px 0;'>{dims_html}</div>
        </div>
        <div style='display:grid; grid-template-columns:1fr 1fr; gap:8px; margin-top:12px; font-size:12px; color:#555;'>
            <div>
                <b>Voix</b><br>
                Pitch: {row['voice_pitch']:.2f} {bar(row['voice_pitch'])}<br>
                Speed: {row['voice_speed']:.2f} {bar(row['voice_speed'], 0.5, 1.5)}<br>
                Energy: {row['voice_energy']:.2f} {bar(row['voice_energy'], 0, 1)}<br>
                Warmth: {row['voice_warmth']:.2f} {bar(row['voice_warmth'], 0, 1)}
            </div>
            <div>
                <b>Visage</b><br>
                Sourcils: {row['face_eyebrows']:.2f} {bar(row['face_eyebrows'])}<br>
                Bouche: {row['face_mouth']:.2f} {bar(row['face_mouth'])}<br>
                Yeux: {row['face_eyes']:.2f} {bar(row['face_eyes'], 0, 1)}<br>
                Regard: {row['face_gaze']:.2f} {bar(row['face_gaze'], 0, 1)}
            </div>
        </div>
    </div>
    """
    display(HTML(html))

# Montrer 5 messages variés
samples = corpus[
    (corpus['testimony_length'] > 80) &
    (corpus['sentiment_score'] > 0.5)
].sample(5, random_state=123)

for _, row in samples.iterrows():
    show_emotion_card(row)

---
## 6. Sauvegarder

In [ ]:
out_dir = Path('../data/processed')

# Dataset complet avec émotions et paramètres avatar
corpus.to_parquet(out_dir / 'corpus_with_emotions.parquet', index=False)
corpus.to_csv(out_dir / 'corpus_with_emotions.csv', index=False)

# Exporter les mappings pour l'application avatar
avatar_config = {
    'voice_map': VOICE_MAP,
    'face_map': FACE_MAP,
    'emotion_dimensions': list(LEXICON.keys()),
    'lexicon': LEXICON,
}
with open(out_dir / 'avatar_config.json', 'w', encoding='utf-8') as f:
    json.dump(avatar_config, f, indent=2, ensure_ascii=False)

# Stats émotionnelles
emotion_stats = {
    'sentiment_distribution': corpus['sentiment_label'].value_counts().to_dict(),
    'dominant_emotions': corpus['dominant_emotion'].value_counts().to_dict(),
    'dominant_dimensions': corpus['dominant_dimension'].value_counts().to_dict(),
    'mean_voice_params': {p: float(corpus[f'voice_{p}'].mean()) for p in ['pitch', 'speed', 'energy', 'warmth']},
    'mean_face_params': {p: float(corpus[f'face_{p}'].mean()) for p in ['eyebrows', 'mouth', 'eyes', 'gaze']},
    'mean_dimensions': {d: float(corpus[f'dim_{d}'].mean()) for d in LEXICON},
}
with open(out_dir / 'emotion_stats.json', 'w', encoding='utf-8') as f:
    json.dump(emotion_stats, f, indent=2, ensure_ascii=False)

print(f'✓ Corpus émotionnel sauvegardé : {len(corpus):,} messages')
print(f'  → corpus_with_emotions.parquet')
print(f'  → corpus_with_emotions.csv')
print(f'  → avatar_config.json')
print(f'  → emotion_stats.json')
print(f'\n🎯 Pipeline complet : MESSAGE → ÉMOTION → VOIX + VISAGE')
print(f'   Chaque message a maintenant :')
print(f'   • Sentiment (positif/négatif/neutre)')
print(f'   • 7 scores d\'émotion (joy, sadness, anger, fear, surprise, disgust, neutral)')
print(f'   • 8 dimensions contextuelles (solidarité, empathie, urgence…)')
print(f'   • 4 paramètres voix (pitch, speed, energy, warmth)')
print(f'   • 4 paramètres visage (sourcils, bouche, yeux, regard)')